# SSSOM Mapping Pipeline

Hybrid retrieval + LLM adjudication

**Inputs**: FAIRe checklist TSV (subject terms) + external term dumps (object terms).
**Outputs**: `faire_mappings.sssom.tsv` conformant to SSSOM v0.15+.

Stages:
1. Load and normalize subject + object term records
2. Hybrid candidate retrieval (lexical + embedding + value-set Jaccard)
3. LLM adjudication with structured JSON output
4. Auto-triage (accept / reject / queue)
<!-- 5. Export review queue for multi-reviewer curation
6. Ingest reviewer decisions and emit SSSOM TSV -->

## Setup

In [1]:
import os, json, re, hashlib, math
from pathlib import Path
from datetime import date
from typing import Optional, Literal
from dataclasses import dataclass, field, asdict

import pandas as pd
import numpy as np
from rapidfuzz import fuzz
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from pydantic import BaseModel, Field, ValidationError

DATA_DIR = Path("./data")
OUT_DIR = Path("./out")
OUT_DIR.mkdir(exist_ok=True)

FAIRE_TSV = DATA_DIR / "FAIRe_checklist_v102.csv"
OBJECT_DUMPS = {
    "DwC":     DATA_DIR / "dwc_recommded_versions.csv",
}

# "MIxS":    DATA_DIR / "mixs_terms.tsv",
# "DNA-derived data extension": DATA_DIR / "gbif_dna_ext_terms.tsv",
# "NMDC":    DATA_DIR / "nmdc_terms.tsv",
# "Humboldt extension": DATA_DIR / "humboldt_terms.tsv",
# "MIQE":    DATA_DIR / "miqe_terms.tsv",
# "INSDC":   DATA_DIR / "insdc_terms.tsv",


/home/camilababo/miniconda3/envs/curategpt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data model

In [2]:
@dataclass
class TermRecord:
    """
    Normalized representation of a term from any source.
    """
    id: str                              # CURIE or URI
    label: str
    definition: str = ""
    synonyms: list[str] = field(default_factory=list)
    data_type: str = ""                  # free text, numeric, Boolean, fixed format, controlled vocabulary
    unit: str = ""
    pattern: str = ""                    # fixed_format spec
    value_set: list[str] = field(default_factory=list)
    parent: str = ""                     # section or ontology parent
    example: str = ""
    source: str = ""                     # DwC | MIxS | NMDC | ...
    source_version: str = ""
    extra: dict = field(default_factory=dict)

    def embedding_text(self) -> str:
        parts = [self.label, self.definition]
        if self.synonyms:
            parts.append("Synonyms: " + "; ".join(self.synonyms))
        if self.value_set:
            parts.append("Values: " + " | ".join(self.value_set[:10]))
        return " . ".join(p for p in parts if p)

## Load FAIRe subject terms

In [3]:
def load_faire(path: Path) -> list[TermRecord]:
    df = pd.read_csv(path, sep="\t", dtype=str).fillna("")
    records = []
    for _, r in df.iterrows():
        value_set = [v.strip() for v in r["controlled_vocabulary_options"].split("|") if v.strip()]
        records.append(TermRecord(
            id=f"faire:{r['term_name']}",
            label=r["term_name"],
            definition=r["description"],
            data_type=r["term_type"],
            unit=r["unit"],
            pattern=r["fixed_format"],
            value_set=value_set,
            parent=r["section"],
            example=r["example"],
            source="FAIReSheets",
            source_version="1.2",
            extra={
                "requirement": r["requirement_level_code"],
                "requirement_condition": r["requirement_level_condition"],
                "sample_type_specificity": r["sample_type_specificity"],
                "declared_source": r["source"],       # HELD OUT — evaluation only
                "declared_uri": r["URI"],             # HELD OUT
                "modifications_made": r["modifications_made"],
                "data_type_scope": r["data_type"],
            },
        ))
    return records

subject_terms = load_faire(FAIRE_TSV)
print(f"Loaded {len(subject_terms)} FAIRe terms")

Loaded 337 FAIRe terms


## Load object terms from each external source

In [ ]:
def load_generic_file(path: Path, source_name: str, sep: str = ",") -> list[TermRecord]:
    """
    Loads a Darwin Core-style terms file (CSV, comma-separated).
    Expected columns: iri, term_localName, label, definition, comments,
    examples, organized_in, issued, status, replaces, rdf_type, term_iri,
    abcd_equivalence, flags.
    Missing columns are tolerated.
    Builds id as "<source_name>:<term_localName>".
    """
    if not path.exists():
        print(f"  [skip] {path} not found")
        return []

    df = pd.read_csv(path, sep=sep, dtype=str).fillna("")

    out = []
    for _, r in df.iterrows():
        local_name = r.get("term_localName", "").strip()
        if not local_name:
            # Skip rows without a local name — no stable id can be built.
            continue

        out.append(TermRecord(
            id=f"{source_name}:{local_name}",
            label=r.get("label", "") or local_name,
            definition=r.get("definition", ""),
            example=r.get("examples", ""),
            source=source_name,
        ))
    return out


object_terms: list[TermRecord] = []
for src, path in OBJECT_DUMPS.items():
    loaded = load_generic_file(path, src, sep=',')
    print(f"  {src}: {len(loaded)} terms")
    object_terms.extend(loaded)
print(f"Total object terms: {len(object_terms)}")

  DwC: 362 terms
Total object terms: 362


## Candidate Retrieval: lexical, embedding, value-set Jaccard

Instead of comparing every subject term against every object term with an LLM, similarity scores are irst computed to identify promising candidates

In [ ]:
EMB_MODEL_NAME = "BAAI/bge-large-en-v1.5"  # swap for 'bge-small-en-v1.5' or base if needed
emb_model = SentenceTransformer(EMB_MODEL_NAME)

def normalize_label(s: str) -> str:
    """
    remove special characters, lower case to decrease formatting differences
    """
    s = s.lower()
    s = re.sub(r"[_\-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def lexical_score(a: TermRecord, b: TermRecord) -> float:
    """
    Computes string similarity, using RadipFuzz to compare two strings by breaking them into tokens.
    """
    la, lb = normalize_label(a.label), normalize_label(b.label)
    scores = [fuzz.token_set_ratio(la, lb)]
    for syn in b.synonyms:
        scores.append(fuzz.token_set_ratio(la, normalize_label(syn)))
    return max(scores) / 100.0

def value_set_jaccard(a: TermRecord, b: TermRecord) -> float:
    """

    """
    if not a.value_set or not b.value_set:
        return 0.0
    A = {v.lower().strip() for v in a.value_set if not v.startswith("other")}
    B = {v.lower().strip() for v in b.value_set if not v.startswith("other")}
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

print("Computing object embeddings...")
obj_texts = [t.embedding_text() for t in object_terms]
obj_embs = emb_model.encode(obj_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

print("Computing subject embeddings...")
subj_texts = [t.embedding_text() for t in subject_terms]
subj_embs = emb_model.encode(subj_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 10174.48it/s]


Computing object embeddings...


Batches: 100%|██████████| 6/6 [00:47<00:00,  7.95s/it]


Computing subject embeddings...


Batches: 100%|██████████| 6/6 [02:13<00:00, 22.25s/it]


## Candidate retrieval per subject term

In [6]:
TOP_K_EACH = 5   # per signal
MAX_CANDIDATES = 15

def retrieve_candidates(subj_idx: int) -> list[tuple[int, dict]]:
    subj = subject_terms[subj_idx]
    subj_emb = subj_embs[subj_idx].reshape(1, -1)

    emb_sims = cosine_similarity(subj_emb, obj_embs)[0]
    emb_top = np.argsort(-emb_sims)[:TOP_K_EACH]

    lex_scores = np.array([lexical_score(subj, o) for o in object_terms])
    lex_top = np.argsort(-lex_scores)[:TOP_K_EACH]

    vs_scores = np.array([value_set_jaccard(subj, o) for o in object_terms])
    vs_top = np.argsort(-vs_scores)[:TOP_K_EACH] if vs_scores.max() > 0 else np.array([], dtype=int)

    cand_ids: dict[int, dict] = {}
    for i in emb_top:
        cand_ids.setdefault(int(i), {})["embedding"] = float(emb_sims[i])
    for i in lex_top:
        cand_ids.setdefault(int(i), {})["lexical"] = float(lex_scores[i])
    for i in vs_top:
        cand_ids.setdefault(int(i), {})["value_set"] = float(vs_scores[i])

    # composite for ranking
    for i, s in cand_ids.items():
        s["composite"] = (
            0.45 * s.get("embedding", 0.0)
            + 0.35 * s.get("lexical", 0.0)
            + 0.20 * s.get("value_set", 0.0)
        )
    ranked = sorted(cand_ids.items(), key=lambda kv: -kv[1]["composite"])[:MAX_CANDIDATES]
    return ranked

# quick sanity check
demo = retrieve_candidates(0)
print(f"\nSubject: {subject_terms[0].label}")
for i, s in demo[:5]:
    print(f"  → {object_terms[i].source}:{object_terms[i].label}  {s}")


Subject: recordedBy
  → DwC:Dataset Name  {'embedding': 0.7100098133087158, 'composite': 0.31950441598892215}
  → DwC:Identification References  {'embedding': 0.7022922039031982, 'composite': 0.31603149175643924}
  → DwC:Previous Identifications  {'embedding': 0.6932528018951416, 'composite': 0.31196376085281374}
  → DwC:Associated Occurrences  {'embedding': 0.6711670756340027, 'composite': 0.3020251840353012}
  → DwC:Associated Sequences  {'embedding': 0.6704328060150146, 'composite': 0.3016947627067566}


In [ ]:
candidate_rows = []

for si, subj in enumerate(subject_terms):
    candidates = retrieve_candidates(si)

    for oi, sig in candidates:

        obj = object_terms[oi]

        candidate_rows.append({
            "subject_id": subj.id,
            "object_id": obj.id,
            "embedding_score": sig.get("embedding", None),
            "lexical_score": sig.get("lexical", None),
            "value_set_score": sig.get("value_set", None),
            "composite_score": sig.get("composite", None),
        })

candidate_df = pd.DataFrame(candidate_rows)
candidate_df.to_csv(OUT_DIR / "candidate_pairs.csv", index=False)

## MapperGPT-style LLM adjudication

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)

LLM_MODEL = "gpt-5"

SKOS_PREDICATES = ["skos:exactMatch", "skos:closeMatch", "skos:broadMatch",
                   "skos:narrowMatch", "skos:relatedMatch", "no_match"]
SYNTACTIC = ["drop_in", "transform", "incompatible"]

class Adjudication(BaseModel):
    predicate: Literal["skos:exactMatch","skos:closeMatch","skos:broadMatch",
                       "skos:narrowMatch","skos:relatedMatch","no_match"]
    confidence: float = Field(ge=0.0, le=1.0)
    syntactic_compatibility: Literal["drop_in","transform","incompatible"]
    justification: str
    transformation_note: Optional[str] = None

PROMPT_TEMPLATE = """You are aligning terms between metadata standards for eDNA and biodiversity data.

SUBJECT (FAIReSheets v1.2):
  ID: {s_id}
  Label: {s_label}
  Definition: {s_def}
  Data type: {s_dtype}
  Unit: {s_unit}
  Pattern: {s_pattern}
  Value set: {s_vs}
  Section: {s_section}
  Example: {s_ex}

CANDIDATE (source: {o_source}):
  ID: {o_id}
  Label: {o_label}
  Definition: {o_def}
  Data type: {o_dtype}
  Unit: {o_unit}
  Pattern: {o_pattern}
  Value set: {o_vs}
  Parent: {o_parent}

Classify the semantic relationship using ONE SKOS predicate:
  skos:exactMatch   — same concept, interchangeable
  skos:closeMatch   — very similar, minor scope/wording differences
  skos:broadMatch   — CANDIDATE is broader than SUBJECT
  skos:narrowMatch  — CANDIDATE is narrower than SUBJECT
  skos:relatedMatch — related but not hierarchical
  no_match          — unrelated

Also classify syntactic compatibility of values:
  drop_in       — identical value patterns and units
  transform     — compatible after conversion (unit, format)
  incompatible  — value patterns fundamentally differ

Respond with ONLY a JSON object, no prose:
{{"predicate": "...", "confidence": 0.0-1.0, "syntactic_compatibility": "...",
  "justification": "<=2 sentences citing specific fields",
  "transformation_note": null or "brief description"}}
"""

def build_prompt(subj: TermRecord, obj: TermRecord) -> str:
    return PROMPT_TEMPLATE.format(
        s_id=subj.id, s_label=subj.label, s_def=subj.definition,
        s_dtype=subj.data_type, s_unit=subj.unit, s_pattern=subj.pattern,
        s_vs=" | ".join(subj.value_set) or "—",
        s_section=subj.parent, s_ex=subj.example,
        o_source=obj.source, o_id=obj.id, o_label=obj.label, o_def=obj.definition,
        o_dtype=obj.data_type, o_unit=obj.unit, o_pattern=obj.pattern,
        o_vs=" | ".join(obj.value_set) or "—", o_parent=obj.parent,
    )

def adjudicate(subj: TermRecord, obj: TermRecord) -> Optional[Adjudication]:
    prompt = build_prompt(subj, obj)

    try:
        response = client.responses.parse(
            model=LLM_MODEL,
            input=prompt,
            text_format=Adjudication,
        )

        return response.output_parsed

    except Exception as e:
        print(f"  [adjudication failed] {subj.label} vs {obj.label}: {e}")
        return None

In [ ]:
# GOOGLE AI STUDIO VER
from google import genai

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

LLM_MODEL = "gemini-2.5-flash"

SKOS_PREDICATES = ["skos:exactMatch", "skos:closeMatch", "skos:broadMatch",
                   "skos:narrowMatch", "skos:relatedMatch", "no_match"]
SYNTACTIC = ["drop_in", "transform", "incompatible"]

class Adjudication(BaseModel):
    predicate: Literal["skos:exactMatch","skos:closeMatch","skos:broadMatch",
                       "skos:narrowMatch","skos:relatedMatch","no_match"]
    confidence: float = Field(ge=0.0, le=1.0)
    syntactic_compatibility: Literal["drop_in","transform","incompatible"]
    justification: str
    transformation_note: Optional[str] = None

PROMPT_TEMPLATE = """You are aligning terms between metadata standards for eDNA and biodiversity data.

SUBJECT (FAIReSheets v1.2):
  ID: {s_id}
  Label: {s_label}
  Definition: {s_def}
  Data type: {s_dtype}
  Unit: {s_unit}
  Pattern: {s_pattern}
  Value set: {s_vs}
  Section: {s_section}
  Example: {s_ex}

CANDIDATE (source: {o_source}):
  ID: {o_id}
  Label: {o_label}
  Definition: {o_def}
  Data type: {o_dtype}
  Unit: {o_unit}
  Pattern: {o_pattern}
  Value set: {o_vs}
  Parent: {o_parent}

Classify the semantic relationship using ONE SKOS predicate:
  skos:exactMatch   — same concept, interchangeable
  skos:closeMatch   — very similar, minor scope/wording differences
  skos:broadMatch   — CANDIDATE is broader than SUBJECT
  skos:narrowMatch  — CANDIDATE is narrower than SUBJECT
  skos:relatedMatch — related but not hierarchical
  no_match          — unrelated

Also classify syntactic compatibility of values:
  drop_in       — identical value patterns and units
  transform     — compatible after conversion (unit, format)
  incompatible  — value patterns fundamentally differ

Respond with ONLY a JSON object, no prose:
{{"predicate": "...", "confidence": 0.0-1.0, "syntactic_compatibility": "...",
  "justification": "<=2 sentences citing specific fields",
  "transformation_note": null or "brief description"}}
"""

def build_prompt(subj: TermRecord, obj: TermRecord) -> str:
    return PROMPT_TEMPLATE.format(
        s_id=subj.id, s_label=subj.label, s_def=subj.definition,
        s_dtype=subj.data_type, s_unit=subj.unit, s_pattern=subj.pattern,
        s_vs=" | ".join(subj.value_set) or "—",
        s_section=subj.parent, s_ex=subj.example,
        o_source=obj.source, o_id=obj.id, o_label=obj.label, o_def=obj.definition,
        o_dtype=obj.data_type, o_unit=obj.unit, o_pattern=obj.pattern,
        o_vs=" | ".join(obj.value_set) or "—", o_parent=obj.parent,
    )

def adjudicate(subj: TermRecord, obj: TermRecord) -> Optional[Adjudication]:
    prompt = build_prompt(subj, obj)

    try:
        response = client.models.generate_content(
            model=LLM_MODEL,
            contents=prompt,
            config={
                "response_mime_type": "application/json",
                "response_schema": Adjudication,
            },
        )

        return response.parsed

    except Exception as e:
        print(f"[adjudication failed] {subj.label} vs {obj.label}: {e}")
        return None


## Run adjudication over all subject terms (with caching)

In [ ]:
CACHE_PATH = OUT_DIR / "adjudication_cache.jsonl"

def cache_key(s: TermRecord, o: TermRecord) -> str:
    return hashlib.md5(f"{s.id}||{o.id}||{o.source}".encode()).hexdigest()

def load_cache() -> dict:
    cache = {}
    if CACHE_PATH.exists():
        for line in CACHE_PATH.read_text().splitlines():
            row = json.loads(line)
            cache[row["key"]] = row["adj"]
    return cache

def append_cache(key: str, adj: dict):
    with CACHE_PATH.open("a") as f:
        f.write(json.dumps({"key": key, "adj": adj}) + "\n")

cache = load_cache()
all_pairs = []   # list of dicts, one per (subject, candidate) pair with adjudication

for si, subj in enumerate(subject_terms):
    candidates = retrieve_candidates(si)
    for oi, sig in candidates:
        obj = object_terms[oi]
        k = cache_key(subj, obj)
        if k in cache:
            adj_dict = cache[k]
        else:
            adj = adjudicate(subj, obj)
            if adj is None:
                continue
            adj_dict = adj.model_dump()
            append_cache(k, adj_dict)
            cache[k] = adj_dict
        all_pairs.append({
            "subject_id": subj.id,
            "subject_label": subj.label,
            "object_id": obj.id,
            "object_label": obj.label,
            "object_source": obj.source,
            "lexical": sig.get("lexical", 0.0),
            "embedding": sig.get("embedding", 0.0),
            "value_set": sig.get("value_set", 0.0),
            "composite": sig["composite"],
            **adj_dict,
        })
    if si % 20 == 0:
        print(f"  processed {si}/{len(subject_terms)} subjects")

pairs_df = pd.DataFrame(all_pairs)
pairs_df.to_csv(OUT_DIR / "all_adjudicated_pairs.tsv", sep="\t", index=False)
print(f"Total adjudicated pairs: {len(pairs_df)}")

You are aligning terms between metadata standards for eDNA and biodiversity data.

SUBJECT (FAIReSheets v1.2):
  ID: faire:recordedBy
  Label: recordedBy
  Definition: A list (concatenated and separated) of names of people responsible for the submitted dataset
  Data type: free text
  Unit: 
  Pattern: 
  Value set: —
  Section: Project
  Example: José E. Crespo

CANDIDATE (source: DwC):
  ID: DwC:type
  Label: Type
  Definition: The nature or genre of the resource.
  Data type: 
  Unit: 
  Pattern: 
  Value set: —
  Parent: 

Classify the semantic relationship using ONE SKOS predicate:
  skos:exactMatch   — same concept, interchangeable
  skos:closeMatch   — very similar, minor scope/wording differences
  skos:broadMatch   — CANDIDATE is broader than SUBJECT
  skos:narrowMatch  — CANDIDATE is narrower than SUBJECT
  skos:relatedMatch — related but not hierarchical
  no_match          — unrelated

Also classify syntactic compatibility of values:
  drop_in       — identical value patter

{
  "predicate": "no_match",
  "confidence": 0.99,
  "syntactic_compatibility": "incompatible",
  "justification": "The SUBJECT 'recordedBy' identifies the people responsible for the submitted dataset, whereas the CANDIDATE 'Type' describes the nature or genre of a resource. Their definitions refer to unrelated metadata concepts and their expected values differ fundamentally (person names versus resource categories).",
  "transformation_note": null
}

## Auto-triage

In [ ]:
AUTO_ACCEPT_CONF = 0.90
AUTO_ACCEPT_LEX  = 0.85
AUTO_REJECT_CONF = 0.85
SPOT_CHECK_FRAC  = 0.10

def triage(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["triage"] = "review"

    accept_mask = (
        (df["predicate"] == "skos:exactMatch")
        & (df["confidence"] >= AUTO_ACCEPT_CONF)
        & (df["lexical"] >= AUTO_ACCEPT_LEX)
    )
    df.loc[accept_mask, "triage"] = "auto_accept"

    # keep only best no_match per subject for rejection
    for sid, grp in df.groupby("subject_id"):
        if (grp["predicate"] == "no_match").all() and (grp["confidence"] >= AUTO_REJECT_CONF).all():
            df.loc[grp.index, "triage"] = "auto_reject"

    # spot-check sample of auto-accepts
    accepted = df[df["triage"] == "auto_accept"]
    if len(accepted) > 0:
        spot = accepted.sample(frac=SPOT_CHECK_FRAC, random_state=42)
        df.loc[spot.index, "triage"] = "spot_check"
    return df

pairs_df = triage(pairs_df)
print(pairs_df["triage"].value_counts())


## Decide later

# ============================================================
# CELL 11 — Build review queue (blind double-review)
# ============================================================
REVIEWERS = ["ana.babo@orcid", "reviewer2@orcid", "reviewer3@orcid"]

def assign_reviewers(df: pd.DataFrame, reviewers: list[str], k: int = 2) -> pd.DataFrame:
    """Deterministic round-robin assignment of k reviewers per row."""
    df = df.copy()
    queue = df[df["triage"].isin(["review", "spot_check"])].reset_index(drop=True)
    n = len(reviewers)
    assignments = []
    for i in range(len(queue)):
        picks = [reviewers[(i + j) % n] for j in range(k)]
        assignments.append(" | ".join(picks))
    queue["assigned_reviewers"] = assignments
    queue["reviewer1_decision"] = ""
    queue["reviewer2_decision"] = ""
    queue["reviewer_notes"] = ""
    # Reorder columns so reviewers see context first, LLM justification, but NOT confidence
    display_cols = ["subject_id","subject_label","object_id","object_label","object_source",
                    "predicate","justification","syntactic_compatibility","transformation_note",
                    "assigned_reviewers","reviewer1_decision","reviewer2_decision","reviewer_notes"]
    return queue[display_cols + [c for c in queue.columns if c not in display_cols]]

review_queue = assign_reviewers(pairs_df, REVIEWERS, k=2)
review_queue.to_csv(OUT_DIR / "review_queue.tsv", sep="\t", index=False)
print(f"Queued {len(review_queue)} pairs for review across {len(REVIEWERS)} reviewers")

# ============================================================
# CELL 12 — Ingest reviewer decisions and compute agreement
# ============================================================
from sklearn.metrics import cohen_kappa_score

def ingest_reviews(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t", dtype=str).fillna("")
    df["agreed"] = df["reviewer1_decision"] == df["reviewer2_decision"]
    df["final_predicate"] = np.where(
        df["agreed"],
        df["reviewer1_decision"],
        "NEEDS_DISCUSSION"
    )
    # kappa (drop empty)
    both = df[(df["reviewer1_decision"] != "") & (df["reviewer2_decision"] != "")]
    if len(both) > 5:
        kappa = cohen_kappa_score(both["reviewer1_decision"], both["reviewer2_decision"])
        print(f"Cohen's kappa (reviewers 1 vs 2): {kappa:.3f}  (n={len(both)})")
    return df

# reviewed = ingest_reviews(OUT_DIR / "review_queue_completed.tsv")

# ============================================================
# CELL 13 — Emit SSSOM TSV
# ============================================================
SSSOM_HEADER = f"""# curie_map:
#   faire: https://example.org/faire/
#   skos: http://www.w3.org/2004/02/skos/core#
#   semapv: https://w3id.org/semapv/vocab/
#   dwc: http://rs.tdwg.org/dwc/terms/
#   mixs: https://w3id.org/mixs/
#   nmdc: https://w3id.org/nmdc/
#   obi: http://purl.obolibrary.org/obo/OBI_
#   envo: http://purl.obolibrary.org/obo/ENVO_
#   uberon: http://purl.obolibrary.org/obo/UBERON_
# mapping_set_id: https://example.org/faire/mappings/v1
# mapping_set_version: 0.1.0
# mapping_set_description: FAIReSheets v1.2 mappings to DwC, MIxS, NMDC, MIQE, Humboldt, INSDC
# license: https://creativecommons.org/licenses/by/4.0/
# mapping_provider: FAIReSheets working group
# mapping_tool: hybrid-retrieval + MapperGPT-style LLM adjudication + double-blind human review
# mapping_date: {date.today().isoformat()}
# extension_definitions:
#   - slot_name: syntactic_compatibility